In [1]:
import numpy as np 
import pandas as pd

In [2]:
np.random.seed(42)
n = 1000 

In [3]:
data = {
    'household_id': range(1, n + 1),
    'income': np.random.lognormal(mean=10.5, sigma=0.6, size=n),
    'wealth': np.random.lognormal(mean=11.5, sigma=1.0, size=n),
    'debt': np.random.exponential(scale=15000, size=n),
    'age': np.random.randint(25, 70, size=n),
    'stock_market_participant': np.random.choice([0, 1], size=n, p=[0.6, 0.4]),
    'liquidity_constrained': np.random.choice([0, 1], size=n, p=[0.7, 0.3]),
    'region_id': np.random.choice([101, 102, 103, 104], size=n) # Regional code (London, SE, etc.)
}

df = pd.DataFrame(data)

In [7]:
median_debt = df['debt'].median()
df['debt_imputed'] = df['debt'].fillna(median_debt)
df['log_income'] = np.log(df['income'])
df['log_wealth'] = np.log(df['wealth'])
df['income_quantile'] = pd.qcut(df['income'], q=5, labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])

In [8]:
# how do equity market participation rates and median wealth vary across income quantiles?

quantile_summary = (
    df.groupby('income_quantile')
    .agg(
        household_count=('household_id', 'count'), 
        mean_income=('income', 'mean'), 
        median_wealth=('wealth', 'median'), 
        stock_participation_rate=('stock_market_participant', 'mean'), 
        liquidity_constrained_share=('liquidity_constrained', 'mean'),
    )
    .reset_index()
)

# compute customized summary statistics across income quantiles
# agg() allows mapping specific statistical functions to individual columns

/var/folders/s4/rcr39x8d50j2rvsrqz_8j0sm0000gn/T/ipykernel_10783/396918258.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby('income_quantile')


In [9]:
print('--- [Step 1] Summary Statistics by Income Quantile ---')
quantile_summary

--- [Step 1] Summary Statistics by Income Quantile ---


,income_quantile,household_count,mean_income,median_wealth,stock_participation_rate,liquidity_constrained_share
0,Q1,200,16879.316358,128998.975272,0.355,0.325
1,Q2,200,26679.958025,110120.527504,0.395,0.280
2,Q3,200,36870.209000,107219.258867,0.420,0.250
3,Q4,200,49724.935058,93402.048505,0.400,0.310
4,Q5,200,89059.032758,96084.379860,0.340,0.310


In [13]:
# within the exact same income quantile, does financial asset accumulaion differ between liquidity constrained and unconstrained households?

cross_summary = df.groupby(['income_quantile', 'liquidity_constrained']).agg(
    mean_wealth = ('wealth', 'mean'),
    median_debt = ('debt_imputed', 'median'),
    stock_participation = ('stock_market_participant', 'mean'),
)


# multi-variable grouping: Income quantile * Liquidity constraintstatus 

/var/folders/s4/rcr39x8d50j2rvsrqz_8j0sm0000gn/T/ipykernel_10783/1350428901.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  cross_summary = df.groupby(['income_quantile', 'liquidity_constrained']).agg(


In [28]:
print('--- [Step 2] Cross-Tabulation: Quantile * Liquidity Constraint ---')
cross_summary

--- [Step 2] Cross-Tabulation: Quantile * Liquidity Constraint ---


mean_wealth   median_debt  \
income_quantile liquidity_constrained                                
Q1              0                      169868.211137   8503.775626   
                1                      209385.041087   8062.039104   
Q2              0                      175089.105151  10006.285172   
                1                      135367.915625   8646.586427   
Q3              0                      179987.550322  11511.462999   
                1                      159360.551220  10692.950245   
Q4              0                      181105.329911   9918.606566   
                1                      187643.009512  14691.300585   
Q5              0                      153587.475278   9210.353808   
                1                      172855.940496  15099.291403   

                                       stock_participation  
income_quantile liquidity_constrained                       
Q1              0                                 0.333333  
                1                                 0.400000  
Q2              0                                 0.381944  
                1                                 0.428571  
Q3              0                                 0.453333  
                1                                 0.320000  
Q4              0                                 0.391304  
                1                                 0.419355  
Q5              0                                 0.384058  
                1                                 0.241935

In [40]:
regional_macro_data = {
    'region_id' : [101, 102, 103, 104],
    'region_name' : ['Greater London', 'South East', 'Midlands', 'North West'],
    'regional_unemployment_rate': [
        0.045,
        0.038,
        0.052,
        0.058,
    ],  # Local unemployment rate
    'regional_house_price_index': [
        125.4,
        118.2,
        105.6,
        102.1,
    ],  # Regional HPI (Base=100)
}

df_regions = pd.DataFrame(regional_macro_data)

# create external regional macroeconomic indicators dataset

In [45]:
print('--- [Step 3] External Regional Dataset ---')
df_regions

--- [Step 3] External Regional Dataset ---


,region_id,region_name,regional_unemployment_rate,regional_house_price_index
0,101,Greater London,0.045,125.4
1,102,South East,0.038,118.2
2,103,Midlands,0.052,105.6
3,104,North West,0.058,102.1


In [46]:
df_merged = pd.merge(df, df_regions, on='region_id', how='left')

# Merge micro household data with macro regional indicators using 'region_id' as the key
# 'how=left' guarantees all primary household observations (N=1000) are retained

In [47]:
print('--- [Step 4] Merged Dataset Structure ---')
print('Dimensions after merge (Rows, Cols):', df_merged.shape)

--- [Step 4] Merged Dataset Structure ---
Dimensions after merge (Rows, Cols): (1000, 16)


In [50]:
# examine whether households residing in regions with lower unemployment and higher property 
# values exhibit systematically higher equity participation rates. 

region_summary = (
    df_merged.groupby('region_name')
    .agg(
        household_count=('household_id', 'count'),
        unemployment_rate=('regional_unemployment_rate', 'first'),
        house_price_index=('regional_house_price_index', 'first'),
        stock_participation_rate=('stock_market_participant', 'mean'),
    )
    .sort_values(by='stock_participation_rate', ascending=False)
)

# Aggregate household outcomes by region to inspect macro-micro relationships

In [51]:
print('--- [Step 5] Regional Level Aggregation Table ---')
region_summary

--- [Step 5] Regional Level Aggregation Table ---


,household_count,unemployment_rate,house_price_index,stock_participation_rate
region_name,,,,
North West,246,0.058,102.1,0.414634
Greater London,262,0.045,125.4,0.389313
Midlands,242,0.052,105.6,0.371901
South East,250,0.038,118.2,0.352000
